# Converting a trained model to tflite
https://www.tensorflow.org/lite/microcontrollers/build_convert#model_conversion

# Convert model to tflite

In [1]:
import tensorflow as tf
import numpy as np

In [2]:
training_spectrogram = np.load('training_spectrogram.npz')
validation_spectrogram = np.load('validation_spectrogram.npz')
test_spectrogram = np.load('test_spectrogram.npz')

X_train = training_spectrogram['X']
X_validate = validation_spectrogram['X']
X_test = test_spectrogram['X']

complete_train_X = np.concatenate((X_train, X_validate, X_test))
# complete_train_X = X_validate

In [3]:
import os

# .keras is a Keras model file/folder, not a SavedModel export directory.
model_candidates = ["fully_trained.model.keras", "trained.model.keras"]
model_path = next((p for p in model_candidates if os.path.exists(p)), None)
if model_path is None:
    raise FileNotFoundError("No .keras model found. Expected one of: fully_trained.model.keras, trained.model.keras")

keras_model = tf.keras.models.load_model(model_path)
converter2 = tf.lite.TFLiteConverter.from_keras_model(keras_model)
converter2.optimizations = [tf.lite.Optimize.DEFAULT]

def representative_dataset_gen():
    for i in range(0, len(complete_train_X), 100):
        # Representative data should be float32 and match model input shape.
        yield [complete_train_X[i:i+100].astype(np.float32)]

converter2.representative_dataset = representative_dataset_gen
converter2.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]

tflite_quant_model = converter2.convert()
open("converted_model.tflite", "wb").write(tflite_quant_model)
print(f"Converted {model_path} -> converted_model.tflite ({len(tflite_quant_model)} bytes)")

INFO:tensorflow:Assets written to: C:\Users\gaharkem\AppData\Local\Temp\tmpnpq3v9ob\assets


INFO:tensorflow:Assets written to: C:\Users\gaharkem\AppData\Local\Temp\tmpnpq3v9ob\assets


Saved artifact at 'C:\Users\gaharkem\AppData\Local\Temp\tmpnpq3v9ob'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 99, 43, 1), dtype=tf.float32, name='input_layer')
Output Type:
  TensorSpec(shape=(None, 5), dtype=tf.float32, name=None)
Captures:
  2275643553552: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2275643554320: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2275643554704: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2275643553744: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2275643554896: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2285420742032: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2285420742608: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2285420743568: TensorSpec(shape=(), dtype=tf.resource, name=None)


c:\Users\gaharkem\ai-voice-control\model\.venv\Lib\site-packages\tensorflow\lite\python\convert.py:846: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(


Converted fully_trained.model.keras -> converted_model.tflite (85328 bytes)


# To convert to C++
This will run a command line too to convert out tflite model into C code.

In [4]:
from pathlib import Path

input_path = Path("converted_model.tflite")
output_path = Path("model_data.cc")

if not input_path.exists():
    raise FileNotFoundError(f"{input_path} was not found. Run the conversion cell first.")

data = input_path.read_bytes()
var_name = "converted_model_tflite"

lines = []
lines.append("#include <cstdint>\n")
lines.append(f"const unsigned char {var_name}[] = {{\n")
for i in range(0, len(data), 12):
    chunk = data[i:i+12]
    hex_values = ", ".join(f"0x{b:02x}" for b in chunk)
    suffix = "," if i + 12 < len(data) else ""
    lines.append(f"  {hex_values}{suffix}\n")
lines.append("};\n")
lines.append(f"const unsigned int {var_name}_len = {len(data)};\n")

output_path.write_text("".join(lines), encoding="utf-8")
print(f"Wrote {output_path} with {len(data)} bytes")

Wrote model_data.cc with 85328 bytes
